# Phase 3 — Silver to Gold Lakehouse

## Purpose
Materialize the Gold star schema and analytics tables using the existing Phase 2 Delta tables.

## Tables created
- `dim_city` — one row per city; deterministic `city_key`.
- `dim_date` — one row per calendar date; `date_key` uses `yyyyMMdd`.
- `fact_weather` — one row per city per hourly timestamp; logical key is `city_key + timestamp`.
- `gold_daily_weather`, `gold_monthly_climate`, `gold_annual_climate`, `gold_extreme_weather`.

## Integration
1. Place this notebook in the same Databricks workspace folder as the Full Load and Incremental Load notebooks.
2. Near the end of each load notebook, run `%run ./10_Silver_to_Gold`.
3. After the load's city-year loop completes, call `run_gold_pipeline(load_type="FULL")` in the full-load notebook and `run_gold_pipeline(load_type="INCREMENTAL")` in the incremental notebook.
4. Do not call Gold inside every city-year loop; refresh once after the loop to avoid repeatedly aggregating all Silver data.

## Design notes
- Uses the table names and weather columns from the uploaded Phase 2 notebooks.
- `observation_year` is a partition column on first creation of `fact_weather` to support time-range filters. If `fact_weather` already exists unpartitioned, inspect/migrate deliberately rather than dropping it.
- Fact uses Delta `MERGE`; incremental runs use the last successful Gold fact-log timestamp as a watermark and merge Silver rows whose `load_timestamp` is newer. A `FULL` run reprocesses the complete Silver baseline. Aggregate tables are rebuilt with idempotent overwrite.
- Extreme-weather thresholds are project-defined indicators, not universal official climate thresholds.
- The logging helper fills `rows_aggregated` / duration fields if present in the existing log schema. Otherwise it records the output count in `rows_inserted` and timing in the existing start/end fields.


In [0]:
# Phase 3 — Silver to Gold Lakehouse
# This notebook defines reusable Gold-layer functions. Call run_gold_pipeline()
# after the Full Load or Incremental Load notebook finishes its Silver processing.

from datetime import datetime
import uuid

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, lit, to_date, date_format, year, month, dayofmonth, quarter,
    when, sequence, explode, xxhash64, broadcast, count, avg, min, max,
    sum as spark_sum
)

spark.conf.set("spark.sql.session.timeZone", "UTC")

BRONZE_TABLE = "bronze_weather"
SILVER_TABLE = "silver_weather"
EXECUTION_LOG_TABLE = "pipeline_execution_logs"
ERROR_LOG_TABLE = "pipeline_error_logs"

DIM_CITY_TABLE = "dim_city"
DIM_DATE_TABLE = "dim_date"
FACT_WEATHER_TABLE = "fact_weather"
DAILY_TABLE = "gold_daily_weather"
MONTHLY_TABLE = "gold_monthly_climate"
ANNUAL_TABLE = "gold_annual_climate"
EXTREME_TABLE = "gold_extreme_weather"

WEATHER_COLUMNS = [
    "temperature_2m", "relative_humidity_2m", "apparent_temperature",
    "precipitation", "rain", "snowfall", "weather_code", "pressure_msl",
    "surface_pressure", "cloud_cover", "wind_speed_10m", "wind_direction_10m",
]

# Project-defined indicators, not official universal climate thresholds.
HOT_DAY_TEMP_C = 30.0
COLD_DAY_TEMP_C = 0.0
HEAVY_RAIN_DAY_MM = 20.0
HEAVY_RAIN_HOUR_MM = 10.0
STRONG_WIND_KMH = 50.0

def table_exists(table_name):
    try:
        return spark.catalog.tableExists(table_name)
    except Exception:
        try:
            spark.table(table_name).limit(1).count()
            return True
        except Exception:
            return False

def write_delta_overwrite(df, table_name, partition_column=None):
    writer = df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    if partition_column:
        writer = writer.partitionBy(partition_column)
    writer.saveAsTable(table_name)

print("Gold configuration loaded. Source:", SILVER_TABLE)


In [0]:
# Gold logging helpers
# These helpers adapt to the existing Phase 2 log-table schema. If rows_aggregated
# or execution_seconds columns exist, they are populated. Otherwise output row
# counts are stored in rows_inserted and timing remains available via start/end.

def log_gold_execution(
    run_id, batch_id, table_name, load_type, start_time, end_time, status,
    rows_read=0, rows_aggregated=0, rows_inserted=0, rows_updated=0,
    error_message=None
):
    if not table_exists(EXECUTION_LOG_TABLE):
        print("WARNING: pipeline_execution_logs not found; Gold log not persisted.")
        return

    seconds = (end_time - start_time).total_seconds()
    values = {
        "run_id": run_id, "batch_id": batch_id, "layer": "Gold",
        "load_type": load_type, "parameter_or_file": table_name,
        "start_time": start_time, "end_time": end_time, "status": status,
        "rows_read": int(rows_read), "rows_aggregated": int(rows_aggregated),
        "rows_inserted": int(rows_inserted), "rows_updated": int(rows_updated),
        "execution_seconds": float(seconds), "duration_seconds": float(seconds),
        "error_message": error_message,
    }
    schema = spark.table(EXECUTION_LOG_TABLE).schema
    record = {field.name: values.get(field.name) for field in schema.fields}
    spark.createDataFrame([record], schema=schema).write.format("delta").mode("append").saveAsTable(EXECUTION_LOG_TABLE)

def log_gold_error(run_id, batch_id, table_name, error):
    if not table_exists(ERROR_LOG_TABLE):
        print("WARNING: pipeline_error_logs not found; Gold error not persisted.")
        return

    values = {
        "error_id": str(uuid.uuid4()), "run_id": run_id, "batch_id": batch_id,
        "layer": "Gold", "source_file": table_name,
        "error_type": type(error).__name__, "error_message": str(error),
        "error_timestamp": datetime.now(),
    }
    schema = spark.table(ERROR_LOG_TABLE).schema
    record = {field.name: values.get(field.name) for field in schema.fields}
    spark.createDataFrame([record], schema=schema).write.format("delta").mode("append").saveAsTable(ERROR_LOG_TABLE)

def run_gold_step(run_id, load_type, table_name, operation, rows_read=0):
    batch_id = f"gold_{table_name}_{run_id[:8]}"
    start_time = datetime.now()
    try:
        result = operation()
        end_time = datetime.now()
        aggregated = int(result.get("rows_aggregated", 0))
        inserted = int(result.get("rows_inserted", aggregated))
        updated = int(result.get("rows_updated", 0))
        log_gold_execution(
            run_id, batch_id, table_name, load_type, start_time, end_time, "SUCCESS",
            rows_read, aggregated, inserted, updated
        )
        print(
            f"{table_name}: SUCCESS | output={aggregated:,} "
            f"| inserted={inserted:,} | updated={updated:,} "
            f"| seconds={(end_time-start_time).total_seconds():.2f}"
        )
        return result
    except Exception as error:
        end_time = datetime.now()
        try:
            log_gold_execution(
                run_id, batch_id, table_name, load_type, start_time, end_time,
                "FAILURE", rows_read, 0, 0, 0, str(error)
            )
            log_gold_error(run_id, batch_id, table_name, error)
        except Exception as log_error:
            print("WARNING: failed to persist Gold error log:", str(log_error))
        raise

print("Gold logging helpers ready.")


In [0]:
# Build dimension tables

def build_dim_city():
    # Bronze preserves the city coordinates and metadata from API ingestion.
    city_source = (
        spark.table(BRONZE_TABLE)
        .select("city", "country", "region", "latitude", "longitude")
        .dropna(subset=["city", "country", "region"])
        .dropDuplicates(["city", "country", "region", "latitude", "longitude"])
    )
    dim_city = (
        city_source
        .withColumn("city_key", xxhash64(col("city"), col("country"), col("region")))
        .select("city_key", "city", "country", "region", "latitude", "longitude")
    )
    rows = dim_city.count()
    if rows != dim_city.select("city_key").distinct().count():
        raise ValueError("dim_city contains duplicate city_key values.")
    write_delta_overwrite(dim_city, DIM_CITY_TABLE)
    return {"rows_aggregated": rows, "rows_inserted": rows, "rows_updated": 0}


def build_dim_date():
    silver = spark.table(SILVER_TABLE)
    bounds = silver.select(
        F.min(to_date(col("timestamp"))).alias("min_date"),
        F.max(to_date(col("timestamp"))).alias("max_date"),
    ).first()
    if bounds["min_date"] is None or bounds["max_date"] is None:
        raise ValueError("Cannot build dim_date: Silver has no valid timestamps.")

    dates = (
        spark.range(1)
        .select(explode(sequence(
            lit(bounds["min_date"]).cast("date"),
            lit(bounds["max_date"]).cast("date")
        )).alias("date"))
        .withColumn("date_key", date_format(col("date"), "yyyyMMdd").cast("int"))
        .withColumn("day", dayofmonth(col("date")))
        .withColumn("month", month(col("date")))
        .withColumn("month_name", date_format(col("date"), "MMMM"))
        .withColumn("quarter", quarter(col("date")))
        .withColumn("year", year(col("date")))
        .withColumn("decade", (F.floor(year(col("date")) / 10) * 10).cast("int"))
        .withColumn(
            "season",
            when(month(col("date")).isin(12, 1, 2), lit("Winter"))
            .when(month(col("date")).isin(3, 4, 5), lit("Spring"))
            .when(month(col("date")).isin(6, 7, 8), lit("Summer"))
            .otherwise(lit("Autumn"))
        )
        .select("date_key", "date", "day", "month", "month_name", "quarter", "year", "decade", "season")
    )
    rows = dates.count()
    if rows != dates.select("date_key").distinct().count():
        raise ValueError("dim_date contains duplicate date_key values.")
    write_delta_overwrite(dates, DIM_DATE_TABLE)
    return {"rows_aggregated": rows, "rows_inserted": rows, "rows_updated": 0}

print("Dimension functions ready.")


In [0]:
# Build and MERGE the hourly fact table

def build_fact_source(updated_since=None):
    silver = spark.table(SILVER_TABLE)
    if updated_since is not None:
        silver = silver.filter(col('load_timestamp') > lit(updated_since))
    city_keys = spark.table(DIM_CITY_TABLE).select("city", "city_key")
    dates = spark.table(DIM_DATE_TABLE).select(
        col("date").alias("observation_date"), "date_key"
    )

    return (
        silver
        .withColumn("observation_date", to_date(col("timestamp")))
        .join(broadcast(city_keys), on="city", how="inner")
        .join(broadcast(dates), on="observation_date", how="inner")
        .withColumn("observation_year", year(col("timestamp")))
        .select(
            "city_key", "date_key", "observation_year", "timestamp",
            *WEATHER_COLUMNS, "source", "batch_id", "load_type", "load_timestamp"
        )
        .dropDuplicates(["city_key", "timestamp"])
    )


def merge_fact_weather(updated_since=None):
    fact_source = build_fact_source(updated_since=updated_since)
    source_rows = fact_source.count()
    if source_rows == 0:
        print('No new or revised Silver rows since the last successful Gold fact run.')
        return {'rows_aggregated': 0, 'rows_inserted': 0, 'rows_updated': 0}

    if not table_exists(FACT_WEATHER_TABLE):
        # Year partitions support common time-range filters.
        write_delta_overwrite(fact_source, FACT_WEATHER_TABLE, "observation_year")
        return {"rows_aggregated": source_rows, "rows_inserted": source_rows, "rows_updated": 0}

    target = DeltaTable.forName(spark, FACT_WEATHER_TABLE)
    compared_columns = ["date_key"] + WEATHER_COLUMNS
    change_condition = "NOT (" + " AND ".join(
        [f"target.{name} <=> source.{name}" for name in compared_columns]
    ) + ")"
    update_columns = (
        ["date_key", "observation_year"] + WEATHER_COLUMNS
        + ["source", "batch_id", "load_type", "load_timestamp"]
    )
    update_values = {name: f"source.{name}" for name in update_columns}

    (
        target.alias("target")
        .merge(
            fact_source.alias("source"),
            "target.city_key = source.city_key "
            "AND target.timestamp = source.timestamp "
            "AND target.observation_year = source.observation_year"
        )
        .whenMatchedUpdate(condition=change_condition, set=update_values)
        .whenNotMatchedInsertAll()
        .execute()
    )
    metrics = target.history(1).select("operationMetrics").first()["operationMetrics"]
    return {
        "rows_aggregated": source_rows,
        "rows_inserted": int(metrics.get("numTargetRowsInserted", 0)),
        "rows_updated": int(metrics.get("numTargetRowsUpdated", 0)),
    }

print("Fact-table MERGE function ready.")


In [0]:
# Daily, monthly, annual and extreme-weather tables

def replace_gold_table(df, table_name):
    rows = df.count()
    write_delta_overwrite(df, table_name)
    return {"rows_aggregated": rows, "rows_inserted": rows, "rows_updated": 0}


def build_gold_daily_weather():
    fact = spark.table(FACT_WEATHER_TABLE)
    daily = (
        fact.groupBy("city_key", "date_key")
        .agg(
            avg("temperature_2m").alias("avg_temperature_c"),
            min("temperature_2m").alias("min_temperature_c"),
            max("temperature_2m").alias("max_temperature_c"),
            spark_sum("precipitation").alias("total_precipitation_mm"),
            spark_sum("rain").alias("total_rain_mm"),
            spark_sum("snowfall").alias("total_snowfall_cm"),
            avg("relative_humidity_2m").alias("avg_relative_humidity_pct"),
            max("wind_speed_10m").alias("max_wind_speed_kmh"),
            max("precipitation").alias("max_hourly_precipitation_mm"),
            spark_sum(when(col("precipitation") >= HEAVY_RAIN_HOUR_MM, 1).otherwise(0)).alias("heavy_rain_hours"),
            spark_sum(when(col("wind_speed_10m") >= STRONG_WIND_KMH, 1).otherwise(0)).alias("strong_wind_hours"),
            count("*").alias("hourly_observations"),
        )
    )
    return replace_gold_table(daily, DAILY_TABLE)


def build_gold_monthly_climate():
    fact = spark.table(FACT_WEATHER_TABLE)
    monthly = (
        fact.withColumn("calendar_year", year(col("timestamp")))
        .withColumn("calendar_month", month(col("timestamp")))
        .groupBy("city_key", "calendar_year", "calendar_month")
        .agg(
            avg("temperature_2m").alias("avg_temperature_c"),
            min("temperature_2m").alias("min_temperature_c"),
            max("temperature_2m").alias("max_temperature_c"),
            spark_sum("precipitation").alias("total_precipitation_mm"),
            spark_sum("rain").alias("total_rain_mm"),
            spark_sum("snowfall").alias("total_snowfall_cm"),
            avg("relative_humidity_2m").alias("avg_relative_humidity_pct"),
            max("wind_speed_10m").alias("max_wind_speed_kmh"),
            count("*").alias("hourly_observations"),
        )
        .withColumn(
            "date_key",
            (col("calendar_year") * 10000 + col("calendar_month") * 100 + lit(1)).cast("int")
        )
        .select(
            "city_key", "date_key", "calendar_year", "calendar_month",
            "avg_temperature_c", "min_temperature_c", "max_temperature_c",
            "total_precipitation_mm", "total_rain_mm", "total_snowfall_cm",
            "avg_relative_humidity_pct", "max_wind_speed_kmh", "hourly_observations"
        )
    )
    return replace_gold_table(monthly, MONTHLY_TABLE)


def build_gold_annual_climate():
    fact = spark.table(FACT_WEATHER_TABLE)
    annual = (
        fact.withColumn("calendar_year", year(col("timestamp")))
        .groupBy("city_key", "calendar_year")
        .agg(
            avg("temperature_2m").alias("avg_temperature_c"),
            min("temperature_2m").alias("min_temperature_c"),
            max("temperature_2m").alias("max_temperature_c"),
            spark_sum("precipitation").alias("total_precipitation_mm"),
            spark_sum("rain").alias("total_rain_mm"),
            spark_sum("snowfall").alias("total_snowfall_cm"),
            avg("relative_humidity_2m").alias("avg_relative_humidity_pct"),
            max("wind_speed_10m").alias("max_wind_speed_kmh"),
            count("*").alias("hourly_observations"),
        )
        .withColumn("date_key", (col("calendar_year") * 10000 + lit(101)).cast("int"))
        .select(
            "city_key", "date_key", "calendar_year",
            "avg_temperature_c", "min_temperature_c", "max_temperature_c",
            "total_precipitation_mm", "total_rain_mm", "total_snowfall_cm",
            "avg_relative_humidity_pct", "max_wind_speed_kmh", "hourly_observations"
        )
    )
    return replace_gold_table(annual, ANNUAL_TABLE)


def build_gold_extreme_weather():
    daily = spark.table(DAILY_TABLE)
    extreme = (
        daily
        .withColumn("hot_day", col("max_temperature_c") >= HOT_DAY_TEMP_C)
        .withColumn("cold_day", col("min_temperature_c") <= COLD_DAY_TEMP_C)
        .withColumn("heavy_rain_day", col("total_precipitation_mm") >= HEAVY_RAIN_DAY_MM)
        .withColumn("has_heavy_rain_hour", col("max_hourly_precipitation_mm") >= HEAVY_RAIN_HOUR_MM)
        .withColumn("has_strong_wind_hour", col("strong_wind_hours") > 0)
        .select(
            "city_key", "date_key", "max_temperature_c", "min_temperature_c",
            "total_precipitation_mm", "max_hourly_precipitation_mm",
            "heavy_rain_hours", "strong_wind_hours", "hot_day", "cold_day",
            "heavy_rain_day", "has_heavy_rain_hour", "has_strong_wind_hour"
        )
    )
    return replace_gold_table(extreme, EXTREME_TABLE)

print("Aggregate-building functions ready.")


In [0]:
# End-to-end Gold refresh entry point
# Call this once after the Full Load or Incremental Load batch loop has completed.

def run_gold_pipeline(load_type="INCREMENTAL"):
    run_id = str(uuid.uuid4())
    silver_rows = spark.table(SILVER_TABLE).count()

    # On incremental runs, merge only new/revised Silver rows since the last
    # successful fact_weather Gold log. FULL runs always rebuild the fact baseline.
    last_gold_success = None
    if load_type.upper() != "FULL" and table_exists(EXECUTION_LOG_TABLE):
        last_gold_success = (
            spark.table(EXECUTION_LOG_TABLE)
            .filter(
                (col("layer") == "Gold")
                & (col("parameter_or_file") == FACT_WEATHER_TABLE)
                & (col("status") == "SUCCESS")
            )
            .agg(F.max("end_time").alias("last_success"))
            .first()["last_success"]
        )

    print("=" * 72)
    print("SILVER → GOLD PIPELINE | run_id:", run_id, "| load_type:", load_type)
    print("Silver rows available:", f"{silver_rows:,}")
    print("=" * 72)

    run_gold_step(run_id, load_type, DIM_CITY_TABLE, build_dim_city, rows_read=silver_rows)
    run_gold_step(run_id, load_type, DIM_DATE_TABLE, build_dim_date, rows_read=silver_rows)
    run_gold_step(
        run_id, load_type, FACT_WEATHER_TABLE,
        lambda: merge_fact_weather(updated_since=last_gold_success),
        rows_read=silver_rows
    )
    run_gold_step(run_id, load_type, DAILY_TABLE, build_gold_daily_weather, rows_read=silver_rows)
    run_gold_step(run_id, load_type, MONTHLY_TABLE, build_gold_monthly_climate, rows_read=silver_rows)
    run_gold_step(run_id, load_type, ANNUAL_TABLE, build_gold_annual_climate, rows_read=silver_rows)
    run_gold_step(run_id, load_type, EXTREME_TABLE, build_gold_extreme_weather, rows_read=silver_rows)

    print("=" * 72)
    print("GOLD REFRESH COMPLETED.")
    print("Fact:", FACT_WEATHER_TABLE)
    print("Dimensions:", DIM_CITY_TABLE, DIM_DATE_TABLE)
    print("Aggregates:", DAILY_TABLE, MONTHLY_TABLE, ANNUAL_TABLE, EXTREME_TABLE)
    print("=" * 72)

print("run_gold_pipeline(load_type='FULL' or 'INCREMENTAL') is ready.")


In [0]:
# Optional optimization — disabled by default
# Run only after Gold tables have been created and validated.
# Support for OPTIMIZE / ZORDER depends on the Databricks runtime and table features.

RUN_GOLD_OPTIMIZE = False

if RUN_GOLD_OPTIMIZE:
    spark.sql(f"OPTIMIZE {FACT_WEATHER_TABLE} ZORDER BY (city_key, date_key)")
    spark.sql(f"OPTIMIZE {DAILY_TABLE} ZORDER BY (city_key, date_key)")
    print("Gold OPTIMIZE commands completed.")
else:
    print("Optional OPTIMIZE skipped. Set RUN_GOLD_OPTIMIZE = True after validation.")
